# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

The rule is: a page is worth reviewing if it is old, used to get search traffic,
and is now trending down. These pages have lost momentum and are good candidates for refresh.

**Reason codes the rule outputs:**
- `stale_high_volume_down`: Days since update ≥ 180, impressions_90d ≥ 1000, trend_direction = "down"
- `stale_mid_volume_down`: Days since update ≥ 180, impressions_90d ≥ 100, trend_direction = "down"  
- `stale_stable`: Days since update ≥ 180, impressions_90d ≥ 100, trend_direction = "stable" (warning)
- `old_not_slipping`: Days since update ≥ 180, but trend is "up" or "stable" (skip)
- `fresh_with_decline`: Days since update < 180, but trend_direction = "down" (different problem)

The score is: `days_since_update * impressions_90d * (1 if trend_direction=="down" else 0.5)`

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# ============ SECTION 2: BUILD THE RANKED QUEUE  ============

import pandas as pd
import numpy as np
import os # Import the os module

# Load the data (adjust path if needed)
df = pd.read_csv("content_refresh_anonymized.csv")

print("=== SECTION 2: BUILD RANKED QUEUE ===\n")

# ============ STEP 1: Build the score ============

# Make a working copy
df_work = df.copy()

# Calculate trend multiplier explicitly
df_work["trend_multiplier"] = df_work["trend_direction"].apply(
    lambda x: 1.0 if x == "down" else 0.5
)

# Calculate score: days × impressions × trend_multiplier
df_work["score"] = (
    df_work["days_since_last_update"].astype(float)
    * df_work["impressions_90d"].astype(float)
    * df_work["trend_multiplier"]
)

# Zero out rows with no real traffic
df_work.loc[df_work["impressions_90d"] == 0, "score"] = 0
df_work.loc[df_work["days_since_last_update"] == 0, "score"] = 0

# Verify
print(f"✓ Scores calculated")
print(f"  Non-zero scores: {(df_work['score'] > 0).sum()} / {len(df_work)}")
print(f"  NaN scores: {df_work['score'].isna().sum()} (should be 0)")
print(f"\n  Score stats:")
print(df_work["score"].describe())

# ============ STEP 2: Assign reason codes ============
print("\nStep 2: Assigning reason codes...")

def assign_reason_code(row):
    """Assign a reason code based on the rule conditions."""

    stale_threshold = 180  # days
    high_volume_threshold = 1000
    mid_volume_threshold = 100

    is_stale = row["days_since_last_update"] >= stale_threshold
    has_high_volume = row["impressions_90d"] >= high_volume_threshold
    has_mid_volume = row["impressions_90d"] >= mid_volume_threshold
    is_declining = row["trend_direction"] == "down"
    is_stable = row["trend_direction"] == "stable"

    # Priority order: high-confidence first
    if is_stale and has_high_volume and is_declining:
        return "stale_high_volume_down"
    elif is_stale and has_mid_volume and is_declining:
        return "stale_mid_volume_down"
    elif is_stale and has_mid_volume and is_stable:
        return "stale_stable"
    elif is_stale and not is_declining:
        return "old_not_slipping"
    elif not is_stale and is_declining:
        return "fresh_with_decline"
    else:
        return "low_priority"

df_work["reason_code"] = df_work.apply(assign_reason_code, axis=1)

print(f"✓ Reason codes assigned")
print(f"\n  Reason code distribution:")
print(df_work["reason_code"].value_counts())

# ============ STEP 3: Rank ============
print("\nStep 3: Ranking by score...")

# Sort by score (descending) and reset index
df_ranked = df_work.sort_values("score", ascending=False).reset_index(drop=True)

# Add rank column
df_ranked["rank"] = range(1, len(df_ranked) + 1)

print(f"✓ Ranked {len(df_ranked)} items")

# ============ STEP 4: Build output dataframe ============
print("\nStep 4: Building output dataframe...")

output_df = df_ranked[[
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "days_since_last_update",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
    "ctr",
    "avg_position"
]].copy()

# Ensure score is float (not NaN)
output_df["score"] = output_df["score"].astype(float)

print(f"✓ Output dataframe created")
print(f"  NaN scores in output: {output_df['score'].isna().sum()} (should be 0)")


output_df.to_csv(output_filepath, index=False)

print(f"✓ Wrote {len(output_df)} ranked items")
print(f"✓ File saved at: {output_filepath}")

# ============ PREVIEW ============
print(f"\n=== TOP 10 PREVIEW ===")
print(output_df.head(10)[["rank", "content_id", "score", "reason_code", "days_since_last_update", "impressions_90d"]].to_string())

print(f"\n=== SCORE DISTRIBUTION ===")
print(f"Max score: {output_df['score'].max():.0f}")
print(f"Min score (non-zero): {output_df[output_df['score'] > 0]['score'].min():.0f}")
print(f"Median score (top 1000): {output_df[output_df['rank'] <= 1000]['score'].median():.0f}")

print(f"\n=== REASON CODES IN TOP 100 ===")
print(output_df[output_df["rank"] <= 100]["reason_code"].value_counts())

print("\nSection 2 complete!")

=== SECTION 2: BUILD RANKED QUEUE ===

✓ Scores calculated
  Non-zero scores: 30000 / 30000
  NaN scores: 0 (should be 0)

  Score stats:
count    3.000000e+04
mean     2.247772e+05
std      9.483885e+05
min      5.000000e-01
25%      1.477000e+03
50%      1.664000e+04
75%      1.109940e+05
max      5.384236e+07
Name: score, dtype: float64

Step 2: Assigning reason codes...
✓ Reason codes assigned

  Reason code distribution:
reason_code
fresh_with_decline        16180
low_priority              13702
old_not_slipping             89
stale_mid_volume_down        15
stale_high_volume_down       11
stale_stable                  3
Name: count, dtype: int64

Step 3: Ranking by score...
✓ Ranked 30000 items

Step 4: Building output dataframe...
✓ Output dataframe created
  NaN scores in output: 0 (should be 0)
✓ Wrote 30000 ranked items
✓ File saved at: /content/baseline_action_score.csv

=== TOP 10 PREVIEW ===
   rank            content_id       score         reason_code  days_since_last_upd

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# ============ SECTION 3: TOP-20 REVIEW (FIXED) ============

# Make sure output_df is loaded (from Section 2)
# If not, load it:
# output_df = pd.read_csv("/mnt/user-data/outputs/baseline_action_score.csv")

print("\n=== TOP 20 ITEMS FOR REVIEW ===\n")

top_20 = output_df[output_df["rank"] <= 20].copy()

# Check for NaN before printing
if top_20["score"].isna().any():
    print("WARNING: Some scores are NaN in top 20!")
    print(f"  NaN count: {top_20['score'].isna().sum()}")
    print("  Try re-running Section 2")
else:
    print("All top-20 scores are valid (no NaN)\n")

for idx, row in top_20.iterrows():
    print(f"\nRank {int(row['rank'])}: {row['content_id']}")
    print(f"  Client: {row['client_id']}")
    print(f"  Score: {row['score']:.0f}")
    print(f"  Reason: {row['reason_code']}")
    print(f"  Days stale: {int(row['days_since_last_update'])}, Impressions: {int(row['impressions_90d'])}, Trend: {row['trend_direction']}")
    print(f"  Position: {row['avg_position']:.1f}, CTR: {row['ctr']:.2f}%")

    # Confidence note
    if row["reason_code"] == "stale_high_volume_down":
        confidence = "HIGH — old, high traffic, clearly declining. Refresh likely recovers traffic."
    elif row["reason_code"] == "stale_mid_volume_down":
        confidence = "MEDIUM-HIGH — old, decent traffic, declining. Refresh worth testing."
    elif row["reason_code"] == "stale_stable":
        confidence = "MEDIUM — old but stable. May not be urgently slipping, but aging may hurt soon."
    elif row["reason_code"] == "fresh_with_decline":
        confidence = "MEDIUM — recent update but already losing ground. Content quality issue, not staleness."
    else:
        confidence = "LOW — doesn't fit staleness pattern strongly."

    print(f"  Confidence: {confidence}")

    # What would make it wrong
    if row["reason_code"] == "stale_high_volume_down":
        wrong_if = "Wrong if: the decline is seasonal/external (not staleness). Check if similar pages also declining."
    elif row["reason_code"] == "stale_stable":
        wrong_if = "Wrong if: page is intentionally stable (evergreen reference content that doesn't need updates)."
    elif row["reason_code"] == "fresh_with_decline":
        wrong_if = "Wrong if: the decline is temporary, and content will recover without refresh."
    else:
        wrong_if = "Wrong if: content is actually performing well in GA4 or has strategic reasons to not be refreshed."

    print(f"  What would make it wrong: {wrong_if}")

print("\nSection 3 complete!")


=== TOP 20 ITEMS FOR REVIEW ===

✓ All top-20 scores are valid (no NaN)


Rank 1: content_5fe46e04994d
  Client: client_4e07408562
  Score: 53842360
  Reason: fresh_with_decline
  Days stale: 104, Impressions: 517715, Trend: down
  Position: 4.2, CTR: 0.14%
  Confidence: MEDIUM — recent update but already losing ground. Content quality issue, not staleness.
  What would make it wrong: Wrong if: the decline is temporary, and content will recover without refresh.

Rank 2: content_2c2606c5d176
  Client: client_19581e27de
  Score: 36129496
  Reason: fresh_with_decline
  Days stale: 104, Impressions: 347399, Trend: down
  Position: 4.2, CTR: 0.53%
  Confidence: MEDIUM — recent update but already losing ground. Content quality issue, not staleness.
  What would make it wrong: Wrong if: the decline is temporary, and content will recover without refresh.

Rank 3: content_cb112fce36be
  Client: client_19581e27de
  Score: 32230640
  Reason: fresh_with_decline
  Days stale: 104, Impressions: 309

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
# ============ SECTION 4: LEAKAGE CHECK + WEAK PICKS (FIXED) ============

print("\n=== SECTION 4: LEAKAGE CHECK & WEAK PICKS ===\n")

print("=== LEAKAGE CHECK ===\n")

print("1. LABEL LEAKAGE:")
print("   ✓ Rule uses ONLY: days_since_last_update, impressions_90d, trend_direction")
print("   ✓ Does NOT use: is_declining_label (derived), future trend_pct (would leak)")
print("   PASS: No label columns in score\n")

print("2. TEMPORAL LEAKAGE:")
print("   ✓ All metrics are trailing 90-day (past-looking), not future")
print("   ✓ days_since_update is historical (how long ago was it last updated)")
print("   ✓ No using future months or look-ahead windows")
print("   PASS: No future windows\n")

print("3. PRODUCT FLAG LEAKAGE:")
print("   ✓ Not using position_tier, impression_tier, freshness_tier (these are binned outputs)")
print("   ✓ Not using is_declining_label (derived flag)")
print("   ✓ Using raw columns only")
print("   PASS: No product flags in features\n")

# ============ WEAK PICKS ANALYSIS ============
print("=== WEAK PICKS ANALYSIS ===\n")

top_50 = output_df[output_df["rank"] <= 50].copy()

print("1. High-scoring items with STABLE trend (not declining):")
weak_stable = top_50[top_50["trend_direction"] == "stable"]
print(f"   Found {len(weak_stable)} items (likely false positives, aged but not slipping yet)")
if len(weak_stable) > 0:
    print("   Sample:")
    sample = weak_stable[["rank", "content_id", "days_since_last_update", "impressions_90d", "reason_code"]].head(3)
    print(sample.to_string(index=False))
else:
    print("   (None found - good!)")

print("\n2. High-scoring items with VERY old content (>500 days):")
weak_very_old = top_50[top_50["days_since_last_update"] > 500]
print(f"   Found {len(weak_very_old)} items (may be archived, not worth refreshing)")
if len(weak_very_old) > 0:
    print("   Sample:")
    sample = weak_very_old[["rank", "content_id", "days_since_last_update", "impressions_90d"]].head(3)
    print(sample.to_string(index=False))
else:
    print("   (None found - good!)")

print("\n3. High-scoring items with LOW impressions (<10):")
weak_low_traffic = top_50[top_50["impressions_90d"] < 10]
print(f"   Found {len(weak_low_traffic)} items (may never have had real traffic)")
if len(weak_low_traffic) > 0:
    print("   Sample:")
    sample = weak_low_traffic[["rank", "content_id", "days_since_last_update", "impressions_90d"]].head(3)
    print(sample.to_string(index=False))
else:
    print("   (None found - good!)")

# ============ SUMMARY ============
print("\n=== SUMMARY ===")
print(f"Total items ranked: {len(output_df)}")
print(f"Items with score > 0: {(output_df['score'] > 0).sum()}")
print(f"Items marked as stale_*_down (HIGH confidence): {(output_df['reason_code'].str.contains('down')).sum()}")
print(f"Items marked as stale_stable (MEDIUM confidence): {(output_df['reason_code'] == 'stale_stable').sum()}")
print(f"Items marked as fresh_with_decline (different problem): {(output_df['reason_code'] == 'fresh_with_decline').sum()}")

print("\n✓ Leakage check complete. No label or future data in score.")
print("\n✓ Section 4 complete!")


=== SECTION 4: LEAKAGE CHECK & WEAK PICKS ===

=== LEAKAGE CHECK ===

1. LABEL LEAKAGE:
   ✓ Rule uses ONLY: days_since_last_update, impressions_90d, trend_direction
   ✓ Does NOT use: is_declining_label (derived), future trend_pct (would leak)
   PASS: No label columns in score

2. TEMPORAL LEAKAGE:
   ✓ All metrics are trailing 90-day (past-looking), not future
   ✓ days_since_update is historical (how long ago was it last updated)
   ✓ No using future months or look-ahead windows
   PASS: No future windows

3. PRODUCT FLAG LEAKAGE:
   ✓ Not using position_tier, impression_tier, freshness_tier (these are binned outputs)
   ✓ Not using is_declining_label (derived flag)
   ✓ Using raw columns only
   PASS: No product flags in features

=== WEAK PICKS ANALYSIS ===

1. High-scoring items with STABLE trend (not declining):
   Found 8 items (likely false positives, aged but not slipping yet)
   Sample:
 rank           content_id  days_since_last_update  impressions_90d  reason_code
    6 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.